In [ ]:
#%pip install streamlit
import soccerdata as sd
import pandas as pd
import streamlit as st

  Using cached streamlit-1.60.0-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.2.2-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.57-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached protobuf-7.35.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached pyarrow-24.0.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached uvicorn-0.52.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached httptools-0.8.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (3.5 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached itsdang

In [12]:
understat = sd.Understat(leagues = "ENG-Premier League", seasons = 2016)
players_seasonstats = understat.read_player_season_stats()

players_seasonstats.head()

[07/30/26 13:12:44] INFO     Saving cached data to /Users/coffeebreakat2pm/soccerdata/data/Understat ]8;id=13318983;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13318984;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

league_id  \
league             season team    player                              
ENG-Premier League 1617   Arsenal Aaron Ramsey                    1   
                                  Ainsley Maitland-Niles          1   
                                  Alex Iwobi                      1   
                                  Alex Oxlade-Chamberlain         1   
                                  Alexis Sánchez                  1   

                                                           season_id  team_id  \
league             season team    player                                        
ENG-Premier League 1617   Arsenal Aaron Ramsey                  2016       83   
                                  Ainsley Maitland-Niles        2016       83   
                                  Alex Iwobi                    2016       83   
                                  Alex Oxlade-Chamberlain       2016       83   
                                  Alexis Sánchez                2016       83   

                                                           player_id position  \
league             season team    player                                        
ENG-Premier League 1617   Arsenal Aaron Ramsey                   504      M S   
                                  Ainsley Maitland-Niles        1750        S   
                                  Alex Iwobi                     500      M S   
                                  Alex Oxlade-Chamberlain        966      M S   
                                  Alexis Sánchez                 498    F M S   

                                                           matches  minutes  \
league             season team    player                                      
ENG-Premier League 1617   Arsenal Aaron Ramsey                  23     1212   
                                  Ainsley Maitland-Niles         1        1   
                                  Alex Iwobi                    26     1471   
                                  Alex Oxlade-Chamberlain       29     1564   
                                  Alexis Sánchez                38     3233   

                                                           goals         xg  \
league             season team    player                                      
ENG-Premier League 1617   Arsenal Aaron Ramsey                 1   3.505757   
                                  Ainsley Maitland-Niles       0        0.0   
                                  Alex Iwobi                   3    3.22738   
                                  Alex Oxlade-Chamberlain      2   2.440089   
                                  Alexis Sánchez              24  17.724456   

                                                           np_goals  \
league             season team    player                              
ENG-Premier League 1617   Arsenal Aaron Ramsey                    1   
                                  Ainsley Maitland-Niles          0   
                                  Alex Iwobi                      3   
                                  Alex Oxlade-Chamberlain         2   
                                  Alexis Sánchez                 22   

                                                               np_xg  assists  \
league             season team    player                                        
ENG-Premier League 1617   Arsenal Aaron Ramsey              3.505757        4   
                                  Ainsley Maitland-Niles         0.0        0   
                                  Alex Iwobi                 3.22738        3   
                                  Alex Oxlade-Chamberlain   2.440089        7   
                                  Alexis Sánchez           15.440944       10   

                                                                 xa  shots  \
league             season team    player                                     
ENG-Premier League 1617   Arsenal Aaron Ramsey              4.43239     42   
                

In [ ]:
def season_label(y: int) -> str:
    """Returns a season label like 2015/16."""
    return f"{y}/{str(y + 1)[-2:]}"


def understat_season_code(y: int) -> str:
    """Returns an explicit Understat season code like 15-16 (avoids ambiguity warnings)."""
    return f"{str(y)[-2:]}-{str(y + 1)[-2:]}"


def players_all_seasons(
    league_list,
    start_year=2015,
    end_year=2025,
    out_file=None,
):
    if out_file is None:
        leagues = (
            league_list.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        out_file = f"{leagues}_players_{start_year}-{str(start_year + 1)[-2:]}_to_{end_year}-{str(end_year + 1)[-2:]}.csv"

    all_dfs = []

    for y in range(start_year, end_year + 1):
        label = season_label(y)
        season_code = understat_season_code(y)
        print(f"Fetching {label} (code={season_code})...")

        try:
            understat = sd.Understat(leagues=league_list, seasons=season_code)
            df = understat.read_player_season_stats().copy()
            all_dfs.append(df)
            print(f"Added {label}: {len(df)} rows")
        except Exception as e:
            print(f"Skipped {label}: {e}")

    if not all_dfs:
        raise ValueError("No season data was collected.")

    combined = pd.concat(all_dfs, ignore_index=False)
    combined.to_csv(out_file, index=True)
    print(f"Saved combined file: {out_file} | total rows: {len(combined)}")
    return combined

In [63]:
leagues = ["ENG-Premier League", "ESP-La Liga", "ITA-Serie A", "GER-Bundesliga", "FRA-Ligue 1"]

for league in leagues:
    players_all_seasons(league_list=league, start_year=2015, end_year=2025)

Fetching 2015/16 (code=15-16)...


[07/30/26 14:24:42] INFO     Saving cached data to /Users/coffeebreakat2pm/soccerdata/data/Understat ]8;id=13320545;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13320546;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

Added 2015/16: 550 rows
Fetching 2016/17 (code=16-17)...


[07/30/26 14:24:43] INFO     Saving cached data to /Users/coffeebreakat2pm/soccerdata/data/Understat ]8;id=13320551;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13320552;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

Added 2016/17: 524 rows
Fetching 2017/18 (code=17-18)...


...

Added 2023/24: 525 rows
Fetching 2024/25 (code=24-25)...


[07/30/26 14:24:52] INFO     Saving cached data to /Users/coffeebreakat2pm/soccerdata/data/Understat ]8;id=13320863;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13320864;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

Added 2024/25: 542 rows
Fetching 2025/26 (code=25-26)...


                    INFO     Saving cached data to /Users/coffeebreakat2pm/soccerdata/data/Understat ]8;id=13320869;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13320870;file:///Users/coffeebreakat2pm/Desktop/my_env/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

Added 2025/26: 553 rows
Saved combined file: fra_ligue_1_players_2015-16_to_2025-26.csv | total rows: 6122
